In [0]:
from pyspark.sql.functions import to_date, col, split

bronze_schemapath = "subscription_pipeline.bronze"
silver_schemapath = "subscription_pipeline.silver"

In [0]:
# Remove Fivetran ingested columns
fivetran_ingested_column_to_remove = [
    "_file",
    "_fivetran_synced",
    "_modified",
    "_line"
]

def remove_fivetran_ingested_column(df):
    for column in fivetran_ingested_column_to_remove:
        df = df.drop(column)
    return df

In [0]:
# Read from bronze and remove metadata columns
df = spark.read.table(f"{bronze_schemapath}.customer")
df_clean = remove_fivetran_ingested_column(df)

# Convert date columns
df_clean = df_clean.withColumn(
    "account_created_date",
    to_date(col("account_created_date"), "dd-MM-yyyy")
)

# Extract customer internal ID from name
df_clean = df_clean.withColumn(
    "customer_internal_id",
    split(col("customer_name"), " ")[1]
)

# Remove duplicates
df_clean = df_clean.dropDuplicates()

# Write to silver
df_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schemapath}.customers")

print(f"✓ Customers table created: {df_clean.count()} rows")